<a href="https://colab.research.google.com/github/mmmmMia-MJ/bio-rag/blob/main/BioRAG_a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 生物医学文献RAG问答助手 v2（PDF上传 + 真实Embedding版）

升级内容：
1. **PDF上传解析**：不再硬编码文本，用户上传PDF文件后自动解析建库
2. **真实Embedding**：用硅基流动 `BAAI/bge-large-zh-v1.5` 替换TF-IDF，理解语义而非字面匹配

运行顺序：1 → 2 → 3 → 4 → 5（问答）或 6（评测集）


## 1. 安装依赖

In [ ]:
!pip install -q pypdf openai numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 5.6 MB/s eta 0:00:00


## 2. 输入硅基流动 API Key

同一个 Key 同时用于：
- **Embedding**（`BAAI/bge-large-zh-v1.5`，把文本变成向量）
- **生成回答**（`Qwen/Qwen2.5-72B-Instruct`）


In [ ]:
import getpass, time
from openai import OpenAI
import numpy as np

api_key = getpass.getpass("请输入你的 硅基流动 API Key: ")
client = OpenAI(api_key=api_key, base_url="https://api.siliconflow.cn/v1")

EMBED_MODEL = "BAAI/bge-large-zh-v1.5"   # 中文语义向量，免费额度内可用
GEN_MODEL   = "Qwen/Qwen2.5-72B-Instruct" # 生成回答

def get_embedding(text: str) -> list[float]:
    """把一段文字变成向量（真实语义embedding，理解词义而非字面重合）"""
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return resp.data[0].embedding

def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

def generate_answer(prompt: str, max_retries: int = 4) -> str:
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=GEN_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=800,
                temperature=0.3,
            )
            answer = resp.choices[0].message.content
            if answer and answer.strip():
                return answer
            time.sleep(5)
        except Exception as e:
            wait = 15
            print(f"  [第{attempt+1}次重试，等待{wait}秒] {e}")
            time.sleep(wait)
    return "【模型多次重试后未返回有效内容】"

print("✅ 客户端初始化完成")


请输入你的 硅基流动 API Key: ··········
✅ 客户端初始化完成


## 3. 上传 PDF 并解析建库

运行这个格子后会出现一个**文件选择按钮**，点击上传你的文献 PDF（可以一次选多个）。
程序会自动：提取文字 → 切块（chunking）→ 调用 embedding API 生成向量 → 存进内存向量库。

**注意**：每篇 PDF 都会调用 embedding API，文件越多花费越多（但极便宜，10篇约几分钱）。


In [ ]:
from google.colab import files
from pypdf import PdfReader
import re
from dataclasses import dataclass
from typing import List

@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    source: str
    text: str
    embedding: list = None  # 真实向量，在build_index()时填入

def extract_text_from_pdf(pdf_bytes: bytes, filename: str) -> str:
    """从PDF字节流中提取纯文本"""
    import io
    reader = PdfReader(io.BytesIO(pdf_bytes))
    text = ""
    for page in reader.pages:
        t = page.extract_text()
        if t:
            text += t + "\n"
    return text.strip()

def chunk_text(text: str, doc_id: str, source: str,
               chunk_size: int = 150, overlap: int = 30) -> List[Chunk]:
    """
    按句子累积切块，保留 overlap 字符的重叠。
    chunk_size=150 控制在 bge-large-zh-v1.5 的 512 token 安全范围内。
    同时切分中英文句子（PDF文献通常中英文混杂）。
    """
    sentences = re.split(r"(?<=[。；.!?\n])", text)
    sentences = [s for s in sentences if s.strip()]
    chunks, buf, idx = [], "", 0
    for sent in sentences:
        if len(buf) + len(sent) > chunk_size and buf:
            chunks.append(Chunk(
                chunk_id=f"{doc_id}_chunk{idx}",
                doc_id=doc_id, source=source, text=buf.strip()
            ))
            idx += 1
            buf = buf[-overlap:] if overlap < len(buf) else buf
        buf += sent
    if buf.strip():
        chunks.append(Chunk(chunk_id=f"{doc_id}_chunk{idx}",
                            doc_id=doc_id, source=source, text=buf.strip()))
    return chunks

# ── 上传入口 ──────────────────────────────────────────────────────────────────
print("请选择你的 PDF 文件（可多选）：")
uploaded = files.upload()

all_chunks: List[Chunk] = []
for filename, content in uploaded.items():
    print(f"\n正在解析：{filename}")
    raw_text = extract_text_from_pdf(content, filename)
    if not raw_text:
        print(f"  ⚠️ 未能提取到文字（可能是扫描版PDF，暂不支持）")
        continue
    doc_id = filename.replace(".pdf", "").replace(" ", "_")[:30]
    chunks = chunk_text(raw_text, doc_id=doc_id, source=filename)
    all_chunks.extend(chunks)
    print(f"  ✅ 提取成功，切分为 {len(chunks)} 个chunk")

print(f"\n共 {len(all_chunks)} 个chunk，开始生成 embedding（调用API）...")

# bge-large-zh-v1.5 的输入限制约为512 token（≈300-400个中文字符）
# 超长chunk直接截断到安全长度，避免触发400 BadRequest
MAX_EMBED_CHARS = 400

long_chunks = [c for c in all_chunks if len(c.text) > MAX_EMBED_CHARS]
if long_chunks:
    print(f"  ⚠️ 发现 {len(long_chunks)} 个超长chunk（>{MAX_EMBED_CHARS}字符），将自动截断")

valid_chunks = []
skip_count = 0
for chunk in all_chunks:
    text = chunk.text.strip()
    # 过滤掉空内容、太短（<10字符）、或者中英文字母+数字比例极低的chunk
    # （通常是PDF里提取出来的乱码、纯符号、页眉页脚等无意义内容）
    chinese_en = sum(1 for c in text if '一' <= c <= '鿿' or c.isalpha())
    if len(text) < 10 or chinese_en < 5:
        skip_count += 1
        continue
    valid_chunks.append(chunk)

print(f"  过滤掉 {skip_count} 个无效chunk（空内容/乱码/过短），剩余 {len(valid_chunks)} 个")
all_chunks = valid_chunks

for i, chunk in enumerate(all_chunks):
    safe_text = chunk.text[:MAX_EMBED_CHARS]
    try:
        chunk.embedding = get_embedding(safe_text)
    except Exception as e:
        print(f"  ⚠️ chunk #{i} ({chunk.chunk_id}) embedding失败，跳过: {e}")
        chunk.embedding = None
        continue
    if (i + 1) % 50 == 0 or i == len(all_chunks) - 1:
        print(f"  进度：{i+1}/{len(all_chunks)}")
    time.sleep(0.3)

# 移除embedding失败的chunk
all_chunks = [c for c in all_chunks if c.embedding is not None]
print(f"\n✅ 向量库构建完成，共 {len(all_chunks)} 个chunk已向量化，可以开始问答了")

print(f"\n✅ 向量库构建完成，共 {len(all_chunks)} 个chunk已向量化，可以开始问答了")


请选择你的 PDF 文件（可多选）：


Saving 1_东北大学本科生毕业设计（论文）-马佳 final.pdf to 1_东北大学本科生毕业设计（论文）-马佳 final.pdf
Saving 3.pdf to 3.pdf
Saving 4.pdf to 4.pdf
Saving 5.pdf to 5 (1).pdf
Saving 8.pdf to 8 (1).pdf
Saving 11.pdf to 11.pdf
Saving 12.pdf to 12.pdf
Saving 14.pdf to 14.pdf
Saving 19.pdf to 19.pdf
Saving 20.pdf to 20.pdf
Saving 34.pdf to 34.pdf
Saving 52.pdf to 52.pdf

正在解析：1_东北大学本科生毕业设计（论文）-马佳 final.pdf
  ✅ 提取成功，切分为 740 个chunk

正在解析：3.pdf
  ✅ 提取成功，切分为 473 个chunk

正在解析：4.pdf
  ✅ 提取成功，切分为 577 个chunk

正在解析：5 (1).pdf
  ✅ 提取成功，切分为 577 个chunk

正在解析：8 (1).pdf
  ✅ 提取成功，切分为 865 个chunk

正在解析：11.pdf
  ✅ 提取成功，切分为 209 个chunk

正在解析：12.pdf
  ✅ 提取成功，切分为 730 个chunk

正在解析：14.pdf
  ✅ 提取成功，切分为 790 个chunk

正在解析：19.pdf
  ✅ 提取成功，切分为 545 个chunk

正在解析：20.pdf
  ✅ 提取成功，切分为 546 个chunk

正在解析：34.pdf
  ✅ 提取成功，切分为 1261 个chunk

正在解析：52.pdf
  ✅ 提取成功，切分为 1727 个chunk

共 9040 个chunk，开始生成 embedding（调用API）...
  过滤掉 22 个无效chunk（空内容/乱码/过短），剩余 9018 个
  进度：50/9018
  进度：100/9018
  进度：150/9018
  进度：200/9018
  进度：250/9018
  进度：300/9018
  进度：350/9018
  进度：400/9018


## 4. RAG核心：真实Embedding检索 + 拒答机制

这一步的逻辑和之前完全一样，只是把"TF-IDF算相似度"换成了"真实向量cosine相似度"：
- **真实Embedding的好处**：能理解语义（比如"磷酸化"和"去磷酸化"方向相反，TF-IDF分不清，embedding能感知）
- **检索逻辑不变**：还是取相似度最高的top_k个chunk塞进prompt
- **拒答机制不变**：相似度低于阈值直接告知"未找到相关信息"


In [ ]:
SYSTEM_PROMPT = (
    "你是一个严谨的生物医学文献问答助手。请严格依据下面提供的【检索到的文献片段】回答问题。\n"
    "规则：\n"
    "1. 只能使用片段中出现的信息作答，不允许使用片段之外的知识编造内容。\n"
    "2. 如果检索到的片段都与问题无关、或者信息不足以回答，必须明确回答"
    "「知识库中没有找到相关信息」，禁止编造。\n"
    "3. 回答时请标注引用的来源文件名。\n"
    "4. 回答请控制在200字以内，条理清晰。\n"
    "5. 专业术语（如PTPN22、14-3-3τ、JAK3、STAT3等）请使用规范写法，不要重复堆砌。\n"
)

def retrieve(query: str, chunks: List[Chunk], top_k: int = 4, sim_threshold: float = 0.4) -> list:
    """
    用真实embedding检索最相关的chunk。
    sim_threshold 从 0.15 提高到 0.4：
    真实embedding的相似度普遍比TF-IDF高，需要相应调高阈值才能起到拒答过滤的效果。
    """
    query_emb = get_embedding(query)
    scored = [(chunk, cosine_sim(query_emb, chunk.embedding)) for chunk in chunks]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

def build_prompt(query: str, retrieved: list, sim_threshold: float = 0.4) -> str:
    useful = [(c, s) for c, s in retrieved if s >= sim_threshold]
    if not useful:
        context_block = "（未检索到任何相关片段）"
    else:
        context_block = "\n\n".join(
            f"[{c.chunk_id}] (来源: {c.source}, 相关度: {s:.3f})\n{c.text}"
            for c, s in useful
        )
    return (
        f"{SYSTEM_PROMPT}\n"
        f"【检索到的文献片段】\n{context_block}\n\n"
        f"【用户问题】\n{query}\n\n"
        f"请基于以上片段作答："
    )

def ask(query: str, top_k: int = 4):
    results = retrieve(query, all_chunks, top_k=top_k)
    prompt = build_prompt(query, results)
    answer = generate_answer(prompt)
    print("【问题】", query)
    print("【检索片段】")
    for c, s in results:
        flag = " (低于阈值，已过滤)" if s < 0.4 else ""
        print(f"  - {c.chunk_id} (score={s:.3f}){flag}")
    print("\n【回答】")
    print(answer)
    print("=" * 60)

print("✅ 检索与问答函数已定义，可以开始提问了")


✅ 检索与问答函数已定义，可以开始提问了


## 5. 开始问答

In [ ]:
# 把下面的问题换成你想问的任何内容
ask("PTPN22如何通过与14-3-3τ的结合影响JAK-STAT信号通路？")


【问题】 PTPN22如何通过与14-3-3τ的结合影响JAK-STAT信号通路？
【检索片段】
  - 1_东北大学本科生毕业设计（论文）-马佳_final_chunk133 (score=0.798)
  - 1_东北大学本科生毕业设计（论文）-马佳_final_chunk338 (score=0.785)
  - 1_东北大学本科生毕业设计（论文）-马佳_final_chunk90 (score=0.764)
  - 1_东北大学本科生毕业设计（论文）-马佳_final_chunk336 (score=0.758)

【回答】
PTPN22通过与14-3-3τ的结合正调控JAK-STAT信号通路。具体而言，当PTPN22蛋白失去与14-3-3τ蛋白的结合能力后，JAK-STAT信号通路中的相关蛋白如JAK3、STAT3以及ERK的磷酸化水平均显著降低，表明PTPN22与14-3-3τ的结合对维持这些蛋白的磷酸化状态和信号通路的活化是必要的。引用来源：1_东北大学本科生毕业设计（论文）-马佳 final.pdf。


In [ ]:
# 可以继续提问，把问题换掉就行
ask("这篇文献的主要实验方法有哪些？")


【问题】 这篇文献的主要实验方法有哪些？
【检索片段】
  - 12_chunk305 (score=0.531)
  - 1_东北大学本科生毕业设计（论文）-马佳_final_chunk200 (score=0.523)
  - 1_东北大学本科生毕业设计（论文）-马佳_final_chunk457 (score=0.522)
  - 12_chunk275 (score=0.520)

【回答】
知识库中没有找到相关信息。提供的文献片段中没有详细描述实验方法。引用来源：12.pdf, 1_东北大学本科生毕业设计（论文）-马佳 final.pdf。


## 6. 评测集（可选）

如果你上传的是毕设相关文献，可以运行下面的评测格子跑24道题，对比v1（TF-IDF）和v2（真实embedding）的准确率差异。
这个"升级前后对比"是简历上最有说服力的迭代数据。


In [ ]:
# -*- coding: utf-8 -*-
"""
评测集（Step 4）
================
24条评测题，分四类：
- A. 单文献事实题（6条）：测基础检索准确度，答案通常在一个chunk里就能找到
- B. 跨文献综合题（8条）：测核心价值——能否把PTPN22→14-3-3τ→JAK-STAT这条机制链
  串联起来回答，这是普通"单篇问答"工具做不到的，也是这个项目最值得讲的能力
- C. 实验证据题（6条）：测能否准确引用具体实验数据（磷酸化水平、表达量变化等）
- D. 拒答题（4条）：测系统能否对知识库外/远缘内容正确拒答，不编造

reference_answer 字段是你判分时参考的标准答案（基于毕设原文内容整理，不是编的）。
判分方式见下方 SCORING_GUIDE。
"""

EVAL_SET = [
    # ===== A. 单文献事实题 =====
    {
        "id": "A1", "category": "单文献事实",
        "question": "PTPN22的C端有几个富含脯氨酸的结构域？分别叫什么？",
        "reference_answer": "4个，分别命名为P1、P2、P3、P4。",
        "source": "doc_3_4",
    },
    {
        "id": "A2", "category": "单文献事实",
        "question": "PTPN22的底物分子有哪些？",
        "reference_answer": "Lck、Zap70、Fyn和TCRζ。",
        "source": "doc_3_4",
    },
    {
        "id": "A3", "category": "单文献事实",
        "question": "PTPN22 R620W变异是什么样的突变？",
        "reference_answer": "Ptpn22 C1858T的单核苷酸多态性，导致人源蛋白第620位点的精氨酸（R）被色氨酸（W）取代。",
        "source": "doc_7_14",
    },
    {
        "id": "A4", "category": "单文献事实",
        "question": "14-3-3蛋白家族一共有几个亚型？",
        "reference_answer": "7个，分别是α、β、γ、δ、ε、ζ和η。",
        "source": "doc_19_20",
    },
    {
        "id": "A5", "category": "单文献事实",
        "question": "人类JAK家族由哪几个基因组成？",
        "reference_answer": "JAK1、JAK2、JAK3和TYK2。",
        "source": "doc_33_38",
    },
    {
        "id": "A6", "category": "单文献事实",
        "question": "PTPN22与14-3-3τ的结合依赖于PTPN22的哪个位点？通过哪个结构域结合？",
        "reference_answer": "依赖于PTPN22第640位丝氨酸（Ser640），通过P2结构域与14-3-3τ特异性结合。",
        "source": "doc_PTPN22_1433_interaction",
    },

    # ===== B. 跨文献综合题（核心能力） =====
    {
        "id": "B1", "category": "跨文献综合",
        "question": "PTPN22如何通过与14-3-3τ的结合影响JAK-STAT信号通路？",
        "reference_answer": (
            "PTPN22通过P2结构域与14-3-3τ结合（依赖Ser640位点），这种互作能正向调控JAK-STAT通路活性："
            "过表达PTPN22-WT会使JAK3、STAT3磷酸化水平上调；当PTPN22失去与14-3-3τ结合能力（S640A突变）后，"
            "JAK3、STAT3磷酸化水平不再上调，说明这一通路调控依赖PTPN22-14-3-3τ的结合。"
        ),
        "source": "doc_PTPN22_1433_interaction + doc_12_15_16 + doc_result_3_3",
    },
    {
        "id": "B2", "category": "跨文献综合",
        "question": "PTPN22单独缺失（不考虑14-3-3τ）对JAK-STAT通路有什么影响？这和PTPN22-14-3-3τ互作被破坏的影响一样吗？",
        "reference_answer": (
            "PTPN22缺失会导致STAT3过度激活、JAK2过度磷酸化（促进自身免疫疾病）；"
            "而PTPN22-14-3-3τ互作被破坏（S640A突变，PTPN22本身还在）则是JAK3、STAT3磷酸化水平显著降低。"
            "两者效果方向相反：完全缺失PTPN22 → JAK-STAT过度激活；PTPN22存在但失去14-3-3τ结合能力 → JAK-STAT激活减弱。"
            "说明PTPN22对JAK-STAT通路的调控作用是双重/依赖于结合状态的，不能简单等同于'PTPN22越多通路越弱'。"
        ),
        "source": "doc_12_15_16 + doc_result_3_3",
    },
    {
        "id": "B3", "category": "跨文献综合",
        "question": "14-3-3τ单独过表达（不和PTPN22结合）会激活JAK-STAT通路吗？",
        "reference_answer": "不会。单独过表达14-3-3τ对JAK3、STAT3、ERK磷酸化水平均没有显著影响，说明14-3-3τ本身不能直接激活JAK-STAT通路，必须通过和PTPN22结合才能发挥调控作用。",
        "source": "doc_result_3_3",
    },
    {
        "id": "B4", "category": "跨文献综合",
        "question": "PTPN22-14-3-3τ互作除了影响JAK-STAT，还通过什么通路影响T细胞活化？",
        "reference_answer": "还通过PI3K/Akt/mTOR信号通路：当PTPN22与14-3-3τ的结合被破坏后，PI3K酶活性受到抑制，进而抑制PI3K/Akt信号通路，影响T细胞活化。",
        "source": "doc_PTPN22_1433_interaction",
    },
    {
        "id": "B5", "category": "跨文献综合",
        "question": "PTPN22致病变异体（如R620W）和PTPN22-14-3-3τ互作机制，是同一回事吗？",
        "reference_answer": (
            "不是同一回事，是两条不同但都涉及PTPN22的机制线：R620W是PTPN22编码基因本身的SNP突变，"
            "影响PTPN22对T/B细胞的负调控功能，与多种自身免疫疾病的遗传风险相关；"
            "而PTPN22-14-3-3τ互作（依赖Ser640位点）是PTPN22通过蛋白互作调控JAK-STAT和PI3K/Akt通路的机制，"
            "两者是PTPN22功能研究里相对独立的两个层面（基因变异 vs 蛋白互作）。"
        ),
        "source": "doc_7_14 + doc_PTPN22_1433_interaction",
    },
    {
        "id": "B6", "category": "跨文献综合",
        "question": "PEP/PTPN22对TCR信号传导是负调控还是正调控？这和它对JAK-STAT通路的调控方向一致吗？",
        "reference_answer": (
            "PTPN22对TCR信号传导本身是负调控（通过去磷酸化Lck、Zap70等信号分子抑制TCR诱导的信号传递）；"
            "但PTPN22（通过与14-3-3τ结合）对JAK-STAT通路是正调控（促进JAK3、STAT3磷酸化）。"
            "两者方向不一致，说明PTPN22在不同信号通路里的调控角色不能一概而论。"
        ),
        "source": "doc_13 + doc_PTPN22_1433_interaction + doc_result_3_3",
    },
    {
        "id": "B7", "category": "跨文献综合",
        "question": "结合JAK-STAT通路的一般机制和PTPN22的研究，PTPN22可能通过哪些方式调节JAK-STAT通路？",
        "reference_answer": (
            "结合两方面信息：JAK-STAT通路本身可以被SOCS、PIAS等分子抑制，也可以被蛋白酪氨酸磷酸酶（PTP）"
            "通过去除JAK、STAT上的磷酸化修饰来调节；PTPN22作为一种PTP，确实可以直接去磷酸化JAKs和STATs，"
            "或者间接通过调节SOCS1、SOCS3等分子来调控JAK-STAT通路活性。"
        ),
        "source": "doc_33_38 + doc_12_15_16",
    },
    {
        "id": "B8", "category": "跨文献综合",
        "question": "PTPN22-S640A突变体和PTPN22-WT相比，在T细胞増殖能力上有什么不同？这个结果和它们对JAK-STAT磷酸化的影响是否吻合？",
        "reference_answer": (
            "过表达PTPN22-WT能显著促进T细胞增殖；而过表达PTPN22 S640A突变体（失去与14-3-3τ结合能力）"
            "增殖反而被抑制。这与磷酸化结果的方向是吻合的——PTPN22-WT能上调JAK3/STAT3磷酸化、促进增殖；"
            "S640A突变体不能维持这种上调，对应表现出增殖被抑制，两组数据相互支持同一个结论："
            "PTPN22需要与14-3-3τ结合才能正向调控JAK-STAT通路并促进T细胞增殖。"
        ),
        "source": "doc_result_3_3 + doc_result_3_5",
    },

    # ===== C. 实验证据题 =====
    {
        "id": "C1", "category": "实验证据",
        "question": "过表达PTPN22-WT对IL-4、IL-22的mRNA表达量有什么影响？",
        "reference_answer": "在CD3/CD28刺激条件下，过表达PTPN22-WT后，IL-4、IL-22的mRNA表达量有所下降。",
        "source": "doc_result_3_4",
    },
    {
        "id": "C2", "category": "实验证据",
        "question": "PTPN22 S640A突变体对IL-4、IL-22表达量的影响和PTPN22-WT相比有什么不同？",
        "reference_answer": "PTPN22-WT会使IL-4、IL-22 mRNA表达量下降；而PTPN22 S640A突变体（不能与14-3-3τ结合）则使IL-4、IL-22表达量上升，方向相反。",
        "source": "doc_result_3_4",
    },
    {
        "id": "C3", "category": "实验证据",
        "question": "PTPN22 S640A突变质粒是如何构建的？",
        "reference_answer": (
            "以pCDNA3.1(+)-PTPN22-3×Flag过表达载体为模板，用反向PCR方法进行点突变，"
            "将1918bp-1920bp的TCC核酸序列转换为GCT序列（即第640位密码子由编码丝氨酸的TCC突变为编码丙氨酸的GCT），"
            "构建完成后通过基因测序验证成功。"
        ),
        "source": "doc_method_construct",
    },
    {
        "id": "C4", "category": "实验证据",
        "question": "检测T细胞增殖能力用的是什么实验方法？",
        "reference_answer": "MTS实验。",
        "source": "doc_result_3_5",
    },
    {
        "id": "C5", "category": "实验证据",
        "question": "过表达PTPN22-S640A突变体对ERK磷酸化水平的影响是什么？",
        "reference_answer": "ERK蛋白磷酸化水平被显著抑制（相比PTPN22-WT只是被微弱抑制）。",
        "source": "doc_result_3_3",
    },
    {
        "id": "C6", "category": "实验证据",
        "question": "JAK2基因突变与哪些疾病有关？",
        "reference_answer": "骨髓增生性肿瘤、淋巴瘤等恶性肿瘤；类风湿性关节炎和银屑病的发生也与JAK-STAT通路异常活化有关。",
        "source": "doc_40_44",
    },

    # ===== D. 拒答题（测试系统不编造） =====
    {
        "id": "D1", "category": "拒答",
        "question": "请介绍一下黑洞的形成机制。",
        "reference_answer": "知识库中没有相关信息，应明确拒答，不应编造内容。",
        "source": "（知识库完全不包含此内容）",
    },
    {
        "id": "D2", "category": "拒答",
        "question": "2024年诺贝尔生理学或医学奖颁给了谁？",
        "reference_answer": "知识库中没有相关信息，应明确拒答。",
        "source": "（知识库完全不包含此内容）",
    },
    {
        "id": "D3", "category": "拒答",
        "question": "类固醇是如何进入线粒体的？",
        "reference_answer": (
            "知识库里确实有这条内容（干扰项doc_distractor_steroid），描述的是类固醇生成性急性调节蛋白（STAR）、"
            "转位蛋白（TSPO）和VDAC1组成的转导体复合物介导胆固醇进入线粒体，14-3-3ε/γ参与调控这一过程。"
            "但这与PTPN22-14-3-3τ-JAK/STAT机制是两件不相关的事，正确回答应该基于这条干扰文献本身的内容回答，"
            "而不应该把它和PTPN22/T细胞信号通路混在一起编造关联。"
        ),
        "source": "distractor_steroid",
    },
    {
        "id": "D4", "category": "拒答",
        "question": "系统性红斑狼疮在不同性别间的发病率有什么差异？",
        "reference_answer": "知识库里有这条内容（干扰项doc_distractor_lupus_epi）：女性发病率远高于男性。这是流行病学层面的描述，与本课题的分子机制研究无关，正确回答应只基于这条内容本身。",
        "source": "distractor_lupus_epi",
    },
]


# ---------- 判分标准 ----------
SCORING_GUIDE = """
判分标准（人工判定，三档）：

✅ 对（1分）：回答覆盖了reference_answer里的关键信息点，没有编造内容
🟡 部分对（0.5分）：覆盖了部分关键信息，但遗漏了重要细节，或表述不够准确
❌ 错（0分）：核心信息错误、张冠李戴（比如把PTPN22-WT和S640A的效果说反），
           或者该拒答时却编造了内容，或者不该拒答时却拒答了

对于B类（跨文献综合题），额外检查一项：
  回答是否真的"串联"了多个文献的信息，而不是只回答了其中一个片段的内容——
  这是检验"跨文献汇总"这个核心功能是否真正发挥作用的关键。

对于D类（拒答题），额外检查：
  D1、D2 应该明确拒答；D3、D4 知识库里确实有相关干扰文献，应该正常回答，
  但不能把干扰文献内容和PTPN22/T细胞机制错误地关联起来。
"""


if __name__ == "__main__":
    from collections import Counter
    cats = Counter(item["category"] for item in EVAL_SET)
    print(f"评测集共 {len(EVAL_SET)} 题，分布：")
    for cat, n in cats.items():
        print(f"  {cat}: {n}题")


评测集共 24 题，分布：
  单文献事实: 6题
  跨文献综合: 8题
  实验证据: 6题
  拒答: 4题


In [ ]:
# 批量跑评测题
results_log = []
for item in EVAL_SET:
    results = retrieve(item["question"], all_chunks, top_k=4)
    prompt = build_prompt(item["question"], results)
    answer = generate_answer(prompt)
    results_log.append({**item, "model_answer": answer})
    print(f"[{item['id']}] {item['question']}")
    print(f"  模型回答: {answer}")
    print(f"  参考答案: {item['reference_answer']}")
    print("-" * 60)
    time.sleep(8)

print(f"\n✅ 全部 {len(results_log)} 题跑完")


[A1] PTPN22的C端有几个富含脯氨酸的结构域？分别叫什么？
  模型回答: PTPN22 的 C 端含有四个与蛋白互作紧密相关的富含脯氨酸的结构域，分别命名为 P1，P2，P3 和 P4。[来源: 1_东北大学本科生毕业设计（论文）-马佳 final.pdf]
  参考答案: 4个，分别命名为P1、P2、P3、P4。
------------------------------------------------------------
[A2] PTPN22的底物分子有哪些？
  模型回答: PTPN22的底物分子包括Lck、Zap70、Fyn 和 TCRζ等。这些底物分子是通过蛋白质芯片技术筛选，并使用质谱分析技术确认与PTPN22相互作用的。来源: 1_东北大学本科生毕业设计（论文）-马佳 final.pdf
  参考答案: Lck、Zap70、Fyn和TCRζ。
------------------------------------------------------------
[A3] PTPN22 R620W变异是什么样的突变？
  模型回答: PTPN22 R620W变异是一种蛋白质变异，具体表现为PTPN22蛋白的第620位氨基酸由精氨酸（R）变为色氨酸（W）。这种变异与T细胞受体（TCR）信号的增强和模式识别受体（PRR）信号的促进有关，同时与Toll样受体（TLR）信号的减弱相关。此外，PTPN22 R620W变异蛋白可能在免疫防御和免疫调节的多个阶段发挥作用。引用来源：52.pdf。
  参考答案: Ptpn22 C1858T的单核苷酸多态性，导致人源蛋白第620位点的精氨酸（R）被色氨酸（W）取代。
------------------------------------------------------------
[A4] 14-3-3蛋白家族一共有几个亚型？
  模型回答: 14-3-3蛋白家族一共有七个亚型，分别是α、β、γ、δ、ε、ζ和η。引用来源: 1_东北大学本科生毕业设计（论文）-马佳 final.pdf (chunk135)
  参考答案: 7个，分别是α、β、γ、δ、ε、ζ和η。
------------------------------------------------------------
[A5]

In [ ]:
## 打分汇总

# 跑完上面那格、看完每道题的"模型回答 vs 参考答案"对照后，
# 把每道题的得分填进下面这个字典：1 = 对，0.5 = 部分对，0 = 错
# 例如：先全填1占位，然后照着打印结果一条条改成你认为对的分数

SCORES = {
    "A1": 1, "A2": 1, "A3": 1, "A4": 1, "A5": 1, "A6": 1,
    "B1": 1, "B2": 1, "B3": 1, "B4": 1, "B5": 1, "B6": 1, "B7": 1, "B8": 1,
    "C1": 1, "C2": 1, "C3": 1, "C4": 1, "C5": 1, "C6": 1,
    "D1": 1, "D2": 1, "D3": 1, "D4": 1,
}

from collections import defaultdict

cat_scores = defaultdict(list)
for item in EVAL_SET:
    cat_scores[item["category"]].append(SCORES[item["id"]])

print("=== 分类准确率 ===")
for cat, scores in cat_scores.items():
    acc = sum(scores) / len(scores) * 100
    print(f"{cat}: {acc:.1f}%  ({sum(scores)}/{len(scores)})")

overall = sum(SCORES.values()) / len(SCORES) * 100
print(f"\n=== 总体准确率: {overall:.1f}% ({sum(SCORES.values())}/{len(SCORES)}) ===")


=== 分类准确率 ===
单文献事实: 100.0%  (6/6)
跨文献综合: 100.0%  (8/8)
实验证据: 100.0%  (6/6)
拒答: 100.0%  (4/4)

=== 总体准确率: 100.0% (24/24) ===
